
# Vertex AI Managed Training
Copied from temporary-scratchpad.ipynb to udate with env vars and remove strings.



In [ ]:
from google.colab import userdata
import os

# Load Project Secrets.
# IMPORTANT: Ensure each secret is activated.
PROJECT_ID = userdata.get("GCP_PROJECT_ID")
GITHUB_USER = userdata.get("GITHUB_USER")
GCS_BUCKET = userdata.get("GCS_BUCKET")
TRAINING_PREFIX = userdata.get("TRAINING_PREFIX")
REPO_NAME = userdata.get("REPO_NAME")
REGION = userdata.get("REGION")

REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
CLONE_PATH = f"/content/{REPO_NAME}"
TRAINING_PACKAGE_NAME = "course9_trainer-0.1.0.tar.gz"
TRAINING_PACKAGE_GCS_URI = f"{TRAINING_PREFIX}/{TRAINING_PACKAGE_NAME}"

## Authenticate in Colab (required)

### Do this once per Colab runtime:

In [ ]:
!gcloud auth login --quiet

## Set PROJECT_ID & REGION

In [ ]:
!gcloud config set project $PROJECT_ID
!gcloud config set compute/region $REGION

## Verify and Confirm USER & PROJECT_ID

**IMPORTANT**: You must see your actual project ID, not [None]

In [ ]:
print("Project ...")
!gcloud config get-value project
print("\nUser ...")
!gcloud config get-value account
print("\nRegion ...")
!gcloud config get-value compute/region
print("\nConfiguration ...")
!gcloud config list project

## Clone Repo Into Colab

**Note**: Clones your GitHub repo (if not already cloned in this Colab session)



In [ ]:
# Remove old clone and download a new one.
# This will always ensure the latest version.
%cd /content
!rm -rf {REPO_NAME}
!git clone {REPO_URL}

## Verify Colab Filesystem

In [ ]:
!pwd
print("\nFolder Contents:\n")
!ls -lh

## Enter Repo Root

Should See:

- training/
- requirements.txt
- setup.py

In [ ]:
%cd {REPO_NAME}
!pwd
!ls -lh

## Build Source Distribution

In [ ]:
!python -m pip install --quiet setuptools wheel
!python setup.py sdist


## Verify Source Distribution

Should see:
- course9_trainer-0.1.0.tar.gz


In [ ]:
!ls dist/

## Upload Training Package to GCS


In [ ]:
!gsutil cp dist/{TRAINING_PACKAGE_NAME} {TRAINING_PREFIX}/

### Verify


In [ ]:
!gsutil ls {TRAINING_PREFIX}/

## Execute Vertex AI Training Job

---
> ⚠️ **Important Note on Vertex AI quotas**
>
> In sandbox, Skills Boost, or newly created projects, this step may fail with
> `RESOURCE_EXHAUSTED: custom_model_training_cpus`.
>
> This is an expected environmental limitation, not a configuration or code error.
> The job definition, packaging, and submission logic below are correct and
> representative of production usage.
---

In [ ]:
!gcloud ai custom-jobs create \
  --region={REGION} \
  --display-name=course9-managed-training \
  --python-package-uris={TRAINING_PACKAGE_GCS_URI} \
  --worker-pool-spec=replica-count=1,machine-type=n1-standard-4,container-image-uri=us-docker.pkg.dev/vertex-ai/training/tf-cpu.2-12.py310:latest \
  --command=python \
  --args=-m,trainer.task

In [ ]:
!gcloud version
!gcloud ai custom-jobs create --help | grep python-package-uris

## Monitor Job

In [ ]:
!gcloud ai custom-jobs list --region={REGION}